# CNN - Clasificación de Radiografías de Tórax (Versión 7)
**Desarrollador:** Levi Flores Vega  
**Mejoras avanzadas incorporadas (NN_extra):** - Capas integradas de **Data Augmentation** para robustez ante rotaciones y traslaciones.
- Capas de **Batch Normalization** antes de las activaciones para acelerar la convergencia y combatir el desvanecimiento del gradiente (*vanishing gradients*).
- **Dropout** balanceado en las etapas convolucionales y el clasificador para controlar el *overfitting*.

1: Importación de Librerías y Configuración de Semillas (Reproducibilidad Absoluta)

In [ ]:
# 1: Importación de Librerías y Configuración de Semillas (Reproducibilidad Absoluta)
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Forzar reproducibilidad absoluta en el entorno
os.environ['PYTHONHASHSEED'] = '42'
np.random.seed(42)
tf.random.set_seed(42)

# Configuración de rutas del proyecto
BASE_DIR = './x-rays'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'validation')
LEADERBOARD_DIR = os.path.join(BASE_DIR, 'leaderboard')
SUBMISSION_PATH = os.path.join(BASE_DIR, 'submission_levi_7.csv')

### 2: Pipeline de Carga Eficiente y Flujo de Datos Optimizado (`tf.data`)
Configuramos la carga desde el directorio en escala de grises (`color_mode='grayscale'`) con dimensiones de 256x256 y un tamaño de lote de 32. Usamos `prefetch` con `AUTOTUNE` para optimizar el rendimiento de la memoria.

In [ ]:
print("--- Inicializando carga de conjuntos de datos ---")

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    color_mode='grayscale',
    image_size=(256, 256),
    batch_size=32,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    color_mode='grayscale',
    image_size=(256, 256),
    batch_size=32,
    shuffle=False
)

class_names = train_ds.class_names
print("\nMapeo oficial detectado para el entrenamiento:", {i: name for i, name in enumerate(class_names)})

y_val = np.concatenate([y for _, y in val_ds], axis=0)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

### 3: Arquitectura de la Red Neuronal Convolucional Avanzada
Implementamos los conceptos clave de las guías de clase:
1. **Data Augmentation en Caliente:** Capas `RandomRotation`, `RandomZoom` y `RandomTranslation`. Keras se encarga automáticamente de aplicarlas **solo en el entrenamiento** y desactivarlas en la inferencia.
2. **Batch Normalization Pre-Activación:** Colocamos `BatchNormalization` inmediatamente antes de la capa `Activation('relu')` para garantizar datos normalizados en la entrada de la no-linealidad. Al hacerlo, desactivamos los sesgos convolucionales (`use_bias=False`) para evitar redundancias de parámetros.
3. **Dropout Progresivo:** Aplicamos una tasa baja (0.1) en las capas iniciales de extracción, una tasa intermedia (0.2) en las profundas, y una tasa restrictiva (0.3) en el clasificador denso final.

In [ ]:
def build_advanced_cnn(input_shape=(256, 256, 1), num_classes=3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # 1. Escalado inicial indispensable para regularizar la entrada
        layers.Rescaling(1./255),
        
        # 2. Data Augmentation controlado en caliente
        layers.RandomRotation(factor=0.05, fill_mode='reflect', seed=42),
        layers.RandomZoom(height_factor=0.05, width_factor=0.05, fill_mode='reflect', seed=42),
        layers.RandomTranslation(height_factor=0.05, width_factor=0.05, fill_mode='reflect', seed=42),
        
        # --- BLOQUE CONVOLUCIONAL 1 ---
        layers.Conv2D(32, (3, 3), padding='same', use_bias=False),
        layers.BatchNormalization(momentum=0.9),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.1, seed=42),
        
        # --- BLOQUE CONVOLUCIONAL 2 ---
        layers.Conv2D(64, (3, 3), padding='same', use_bias=False),
        layers.BatchNormalization(momentum=0.9),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.1, seed=42),
        
        # --- BLOQUE CONVOLUCIONAL 3 ---
        layers.Conv2D(64, (3, 3), padding='same', use_bias=False),
        layers.BatchNormalization(momentum=0.9),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.15, seed=42),
        
        # --- BLOQUE CONVOLUCIONAL 4 ---
        layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
        layers.BatchNormalization(momentum=0.9),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2, seed=42),
        
        # --- CLASIFICADOR DENSO ROBUSTO ---
        layers.Flatten(),
        layers.Dense(128, use_bias=False),
        layers.BatchNormalization(momentum=0.9),
        layers.Activation('relu'),
        layers.Dropout(0.25, seed=42),
        
        layers.Dense(num_classes, activation=None)
    ])
    return model

model = build_advanced_cnn()
model.summary()

### 4: Compilación y Entrenamiento con Control de Regularización Dinámica
Compilamos utilizando la pérdida robusta `SparseCategoricalCrossentropy(from_logits=True)`. Además, añadimos:
- `EarlyStopping`: detiene el entrenamiento si la pérdida de validación deja de mejorar durante 8 épocas y recupera los mejores pesos.
- `ReduceLROnPlateau`: reduce el `learning_rate` a la mitad si el modelo se estanca en una meseta durante 3 épocas.

In [ ]:
print("\n--- Compilando y entrenando el modelo optimizado ---")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,          # Mayor margen de convergencia
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks_list,
    verbose=1
)

### 5: Evaluación de Métricas y Diagnóstico del Rendimiento
Calculamos las métricas detalladas en el conjunto de validación utilizando `classification_report` de Scikit-Learn (precisión, recall, F1-score por clase) y visualizamos la matriz de confusión.

In [ ]:
print("\n--- Generando Reporte de Clasificación sobre Conjunto de Validación ---")

logits_preds = model.predict(val_ds, verbose=0)
prob_preds = tf.nn.softmax(logits_preds).numpy()
y_pred = np.argmax(prob_preds, axis=1)

print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_val, 
    y_pred, 
    display_labels=class_names, 
    cmap='Blues', 
    ax=ax,
    colorbar=False
)
plt.title("Matriz de Confusión Avanzada (Muestra Validación)")
plt.grid(False)
plt.show()

### 6: Función Predictiva Definitiva para el Leaderboard y Envío de Respuestas
Esta función procesa las imágenes de la carpeta del leaderboard aplicando el ordenamiento correcto y extrayendo los identificadores exactos. Ejecuta las predicciones de forma completamente estricta en escala de grises a 256x256 y genera el archivo CSV final listo para la entrega.

In [ ]:
def generar_submission_ganadora(modelo, carpeta_leaderboard, ruta_csv):
    print(f"\n--- Iniciando proceso de inferencia en: {carpeta_leaderboard} ---")
    
    archivos = [f for f in os.listdir(carpeta_leaderboard) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    archivos.sort(key=lambda x: int(re.findall(r'\d+', x)[0]) if re.findall(r'\d+', x) else x)
    
    ids_lista = []
    pred_lista = []
    
    for archivo in archivos:
        id_numerico = int(re.findall(r'\d+', archivo)[0])
        ruta_img = os.path.join(carpeta_leaderboard, archivo)
        
        img = tf.keras.utils.load_img(ruta_img, color_mode='grayscale', target_size=(256, 256))
        img_array = tf.keras.utils.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)
        
        # Predicción limpia
        logits = modelo.predict(img_array, verbose=0)
        probs = tf.nn.softmax(logits).numpy()
        clase_predicha = np.argmax(probs, axis=1)[0]
        
        ids_lista.append(id_numerico)
        pred_lista.append(clase_predicha)
        
    df_sub = pd.DataFrame({'id': ids_lista, 'prediction': pred_lista})
    df_sub.to_csv(ruta_csv, index=False)
    print(f"¡CSV definitivo generado con éxito en: {ruta_csv}! Total filas: {len(df_sub)}")
    return df_sub

df_entrega = generar_submission_ganadora(model, LEADERBOARD_DIR, SUBMISSION_PATH)